# Model 3: Rider Detector
Detects motorcycle riders and enables triple-riding enforcement.

### Quick Instructions
- **In Kaggle**: Set accelerator to **GPU T4 x2** or **GPU P100**.
- **In Colab**: Set accelerator to **T4 GPU** (Runtime -> Change runtime type).
- Run all cells sequentially. Once finished, `rider_best.pt` will be ready for direct download!

In [ ]:
!pip install -q ultralytics

import os, sys, glob, yaml, shutil, zipfile
from pathlib import Path
import torch
from ultralytics import YOLO

# Strict environment detection
IS_KAGGLE = Path('/kaggle').exists()
IS_COLAB = 'google.colab' in sys.modules or (Path('/content').exists() and not IS_KAGGLE)

if IS_KAGGLE:
    BASE_DIR = Path('/kaggle/working')
    PLATFORM = 'Kaggle'
elif IS_COLAB:
    BASE_DIR = Path('/content')
    PLATFORM = 'Colab'
else:
    BASE_DIR = Path('.').resolve()
    PLATFORM = 'Local'

RUNS_DIR = BASE_DIR / 'runs'
YAMLS_DIR = BASE_DIR / 'yamls'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
YAMLS_DIR.mkdir(parents=True, exist_ok=True)

has_cuda = torch.cuda.is_available()
DEVICE = 0 if has_cuda else 'cpu'
print('=' * 50)
print(f'Platform Detected: {PLATFORM}')
print(f'Base Directory   : {BASE_DIR}')
print(f'CUDA Available   : {has_cuda}')
if has_cuda:
    print(f'GPU Device Name  : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: Running on CPU! Please enable GPU accelerator.')
print('=' * 50)


In [ ]:
# 1. Extract zip if present in environment
for z in list(BASE_DIR.glob('*.zip')) + list(Path('.').glob('*.zip')):
    if any(k in z.name.lower() for k in ['two-wheeler', 'violation', 'dataset', 'helmet']):
        print(f'Unpacking archive: {z.name}...')
        with zipfile.ZipFile(z, 'r') as ref:
            ref.extractall(BASE_DIR / 'dataset')
        break

# 2. Find dataset.yaml for rider_train_val
candidate_roots = [BASE_DIR / 'dataset', BASE_DIR, Path('/kaggle/input'), Path('.')]
target_yaml = None
for p in candidate_roots:
    candidates = list(p.rglob('*rider_train_val*/**/dataset.yaml')) + [f for f in p.rglob('dataset.yaml') if 'rider_train_val' in str(f)]
    if candidates:
        target_yaml = candidates[0]
        break

assert target_yaml is not None, 'Could not find dataset.yaml for rider_train_val! Ensure dataset is attached.'
print(f'Found target YAML: {target_yaml}')

# 3. Build standalone verified YAML pointing to absolute paths
root = target_yaml.parent
cfg = yaml.safe_load(open(target_yaml, encoding='utf-8'))
train_p = root / cfg.get('train', 'train/images')
val_p = root / cfg.get('val', 'val/images')
if not train_p.exists(): train_p = root / 'train' / 'images'
if not val_p.exists(): val_p = root / 'val' / 'images'

clean_cfg = {
    'path': '',
    'train': str(train_p).replace('\\', '/'),
    'val': str(val_p).replace('\\', '/'),
    'nc': len(cfg['names']),
    'names': cfg['names']
}
custom_yaml = YAMLS_DIR / 'rider_train_val.yaml'
with open(custom_yaml, 'w', encoding='utf-8') as f:
    yaml.dump(clean_cfg, f, sort_keys=False)
print(f'Verified YAML ready: {custom_yaml}')


In [ ]:
# Train YOLOv8 for Model 3: Rider Detector
model = YOLO('yolov8s.pt')
_ = model.train(
    data         = str(custom_yaml),
    epochs       = 80 if has_cuda else 3,
    imgsz        = 640,
    batch        = 16 if has_cuda else 2,
    device       = DEVICE,
    patience     = 20,
    project      = str(RUNS_DIR),
    name         = 'rider_train_val',
    exist_ok     = True,
    plots        = True
)

best_weights = str(model.trainer.best)
print(f'Training complete! Best weights saved at: {best_weights}')


In [ ]:
# Validate Model 3: Rider Detector
val_model = YOLO(best_weights)
metrics = val_model.val(data=str(custom_yaml), imgsz=640, device=DEVICE)

print('\n' + '=' * 60)
print(' EVALUATION REPORT: MODEL 3: RIDER DETECTOR')
print('=' * 60)
print(f'  Overall Precision (P) : {metrics.box.mp:.4f}  ({metrics.box.mp*100:.2f}%)')
print(f'  Overall Recall (R)    : {metrics.box.mr:.4f}  ({metrics.box.mr*100:.2f}%)')
print(f'  mAP@50 (IoU=0.50)     : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.2f}%)')
print(f'  mAP@50-95 (Strict)    : {metrics.box.map:.4f}  ({metrics.box.map*100:.2f}%)')
print('-' * 60)
print('  Per-Class Accuracy Breakdown:')
for i, cls_idx in enumerate(metrics.box.ap_class_index):
    c_name = val_model.names[cls_idx]
    c_map50 = metrics.box.all_ap[i][0] if hasattr(metrics.box, 'all_ap') else metrics.box.maps[i]
    print(f'    • Class [{c_name:15s}]: mAP50 = {c_map50:.4f}')
print('-' * 60)
speed = metrics.speed
inf_ms = speed.get('inference', 13.0)
fps = 1000.0 / inf_ms if inf_ms > 0 else 0
print(f'  Inference Speed       : {inf_ms:.2f} ms per frame (~{fps:.1f} FPS)')
print('=' * 60)


In [ ]:
# Export and Download rider_best.pt
export_path = BASE_DIR / 'rider_best.pt'
shutil.copy(best_weights, export_path)
print(f'Exported to: {export_path} ({export_path.stat().st_size / (1024*1024):.1f} MB)')

if PLATFORM == 'Colab':
    try:
        from google.colab import files
        print('Initiating browser download for rider_best.pt...')
        files.download(str(export_path))
    except Exception as e:
        print('In Colab: Download manually from left files sidebar (/content/)')
elif PLATFORM == 'Kaggle':
    from IPython.display import FileLink, display
    print('\nDownload from Kaggle:')
    print('1. Click the download link below directly in this cell:')
    display(FileLink('rider_best.pt'))
    print('2. Or check the right-hand Output sidebar -> rider_best.pt')

print('\nDrop this file into: Automatic-Helmet-Detection/weights/rider_best.pt')
